In [9]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 1 - WEEK 11 BAYESIAN OPTIMISATION
# Run from inside week11/
# ============================================================
#
# IMPORTANT:
# Function 1 is modelled on the RAW objective.
#
# We apply only a POSITIVE LINEAR scaling:
#
#     Y_scaled = Y / max(abs(Y))
#
# This preserves:
# - ordering
# - argmax
# - maximisation objective
#
# The GP is treated primarily as a LOCAL RANKING MODEL because
# its absolute predictive scale has repeatedly been unreliable.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 11 data
# ------------------------------------------------------------

X = np.load("function1/initial_inputs.npy")
Y = np.load("function1/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 10 calibration check
# ------------------------------------------------------------
#
# Week 10 selected:
# [0.75730209, 0.73556915]
#
# Prior GP prediction:
# raw mean ≈ 9.7498e-06
# raw std  ≈ 1.41677e-05
#
# Actual submitted result:
# -1.1724650608954473e-20
#
# The standardised residual may not look extreme because the
# predicted uncertainty was huge relative to the true optimum.
# Absolute calibration remains poor.
# ------------------------------------------------------------

week10_pred_mean_raw = 9.7498e-06
week10_pred_std_raw = 1.41677e-05
week10_actual = -1.1724650608954473e-20

week10_error = (
    week10_actual
    - week10_pred_mean_raw
)

week10_z_error = (
    week10_error
    / week10_pred_std_raw
)

print("\n================================")
print("WEEK 10 CALIBRATION CHECK")
print("================================")

print("Predicted raw mean:", week10_pred_mean_raw)
print("Predicted raw std :", week10_pred_std_raw)
print("Actual            :", week10_actual)

print("\nPrediction error:")
print(week10_error)

print("\nError / predicted std:")
print(week10_z_error)


# ------------------------------------------------------------
# 3. Positive linear scaling
# ------------------------------------------------------------

y_scale = np.max(
    np.abs(Y)
)

Y_scaled = (
    Y / y_scale
)

best_y_scaled = (
    best_y / y_scale
)

print("\n================================")
print("POSITIVE LINEAR Y SCALING")
print("================================")

print("Scale:")
print(y_scale)

print("\nBest scaled Y:")
print(best_y_scaled)

print(
    "\nArgmax preserved:",
    np.argmax(Y) == np.argmax(Y_scaled)
)


# ------------------------------------------------------------
# 4. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-4, 1e2)
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.1,
        length_scale_bounds=(0.005, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y_scaled
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 5. Empirical local scale
# ------------------------------------------------------------
#
# Search remains centred on the ACTUAL incumbent.
#
# Week 10 produced another very strong near-zero observation
# nearby, so use the nearest observed distance to determine
# a conservative empirical trust-region size.
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask] - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

empirical_cap = min(
    1.25 * nearest_distance,
    0.05
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print("Nearest observed point:")
print(nearest_distance)

print("\nEmpirical cap:")
print(empirical_cap)


# ------------------------------------------------------------
# 6. ARD trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.20 * lengthscales,
    0.008,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("WEEK 11 TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(trust_half_width)

print("\nLower:")
print(lower)

print("\nUpper:")
print(upper)


# ------------------------------------------------------------
# 7. Generate dense trust-region candidates
# ------------------------------------------------------------

rng = np.random.default_rng(42)

candidates = rng.uniform(
    lower,
    upper,
    size=(350000, 2)
)

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 8. Predictions
# ------------------------------------------------------------

mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

mu_raw = (
    mu_scaled * y_scale
)

sigma_raw = (
    sigma_scaled * y_scale
)


# ------------------------------------------------------------
# 9. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])

print(
    "scaled mean =",
    mu_scaled[ei_idx]
)

print(
    "scaled std =",
    sigma_scaled[ei_idx]
)

print(
    "raw mean =",
    mu_raw[ei_idx]
)

print(
    "raw std =",
    sigma_raw[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)


# ------------------------------------------------------------
# 10. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu_scaled
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "scaled mean =",
    mu_scaled[mean_idx]
)

print(
    "scaled std =",
    sigma_scaled[mean_idx]
)

print(
    "raw mean =",
    mu_raw[mean_idx]
)

print(
    "raw std =",
    sigma_raw[mean_idx]
)


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n scaled mean =", mu_scaled[idx],
        "\n scaled std =", sigma_scaled[idx],
        "\n raw mean =", mu_raw[idx],
        "\n raw std =", sigma_raw[idx],
        "\n UCB =", UCB[idx],
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from actual incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Trust-region boundary diagnostic
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(len(x)):

        if (
            abs(x[j] - lower[j])
            <= tol
        ):
            status.append(
                f"x{j+1}=LOWER"
            )

        elif (
            abs(x[j] - upper[j])
            <= tol
        ):
            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )

DATA
X shape: (20, 2)
Y shape: (20,)

Current best:
[0.73102363 0.73299988] -> 7.710875114502849e-16

Y range:
min = -0.0036060626443634764
max = 7.710875114502849e-16
std = 0.0007859230825591007

WEEK 10 CALIBRATION CHECK
Predicted raw mean: 9.7498e-06
Predicted raw std : 1.41677e-05
Actual            : -1.1724650608954473e-20

Prediction error:
-9.749800000000011e-06

Error / predicted std:
-0.6881709804696606

POSITIVE LINEAR Y SCALING
Scale:
0.0036060626443634764

Best scaled Y:
2.1383086970370515e-13

Argmax preserved: True


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.253**2 * Matern(length_scale=[2, 0.0366], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[2.         0.03662297]

Normalised inverse-lengthscale sensitivity:
[0.0179822 0.9820178]

EMPIRICAL LOCAL SCALE
Nearest observed point:
0.026278458561148094

Empirical cap:
0.03284807320143512

WEEK 11 TRUST REGION
Centre:
[0.73102363 0.73299988]

Half-widths:
[0.03284807 0.008     ]

Lower:
[0.69817556 0.72499988]

Upper:
[0.7638717  0.74099988]

Candidates after duplicate filtering:
180948

PRIMARY EI
candidate = [0.6985285  0.74099267]
scaled mean = -0.001180963348408337
scaled std = 0.004629837872498579
raw mean = -4.258627815057713e-06
raw std = 1.6695485401476397e-05
EI = 0.0013163206286793772

HIGHEST PREDICTED MEAN
candidate = [0.69818713 0.73417334]
scaled mean = 0.0005870337324664354
scaled std = 0.0011475063647506329
raw mean = 2.1168804136284756e-06
raw std = 4.137979836096587e-06

UCB DIAGNOSTICS

beta=0.05 
 candidate = [0.69818713 0.73417334] 

In [10]:
# ============================================================
# FINAL FUNCTION 1 - WEEK 11 SELECTION
# ============================================================
#
# Absolute GP predictions remain poorly calibrated for F1,
# so the GP is used primarily as a local ranking model.
#
# Highest posterior mean and UCB beta = 0.05, 0.1, 0.25
# and 0.5 all select exactly the same candidate.
#
# EI and beta = 1.0 are more uncertainty-driven.
#
# The selected point lies on the x1 trust-region boundary,
# but we deliberately do NOT expand further.

final_idx = np.argmax(mu_scaled)

week11_candidate = candidates[final_idx]

print("Week 11 Function 1 candidate:")
print(week11_candidate)

print("\nPredicted scaled mean:")
print(mu_scaled[final_idx])

print("\nPredicted scaled std:")
print(sigma_scaled[final_idx])

print("\nPredicted raw mean:")
print(mu_raw[final_idx])

print("\nPredicted raw std:")
print(sigma_raw[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week11_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week11_candidate
)

print("\nPortal format:")
print(portal)

Week 11 Function 1 candidate:
[0.69818713 0.73417334]

Predicted scaled mean:
0.0005870337324664354

Predicted scaled std:
0.0011475063647506329

Predicted raw mean:
2.1168804136284756e-06

Predicted raw std:
4.137979836096587e-06

Distance from current best:
0.032857457084411096

Portal format:
0.698187-0.734173
